<a href="https://colab.research.google.com/github/mirzafani808-source/neurofive-ml-track/blob/main/titanic-survival-prediction/titanic_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!wget -O titanic.csv https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv


--2026-07-21 11:05:51--  https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 60302 (59K) [text/plain]
Saving to: ‘titanic.csv’

titanic.csv         100%[===================>]  58.89K  --.-KB/s    in 0.01s   

2026-07-21 11:05:51 (4.73 MB/s) - ‘titanic.csv’ saved [60302/60302]



In [7]:
!wget -O titanic.csv https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv

--2026-07-21 11:05:51--  https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 60302 (59K) [text/plain]
Saving to: ‘titanic.csv’

titanic.csv         100%[===================>]  58.89K  --.-KB/s    in 0.01s   

2026-07-21 11:05:51 (4.63 MB/s) - ‘titanic.csv’ saved [60302/60302]



In [8]:
!ls

sample_data  titanic.csv


In [9]:
"""
Model Evaluation & Tuning: Beyond Accuracy
--------------------------------------------------------
Revisits the Titanic Logistic Regression model, evaluates it with
Precision/Recall/F1, tunes hyperparameters with GridSearchCV,
and compares before/after performance.
"""

import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score, precision_score, recall_score

# ---------------------------------------------------------
# 1. Load & clean data (same as previous task)
# ---------------------------------------------------------
df = pd.read_csv("titanic.csv")

df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
df = pd.get_dummies(df, columns=["Sex", "Embarked"], drop_first=True)

X = df.drop(columns=["Survived"])
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---------------------------------------------------------
# 2. BASELINE model (from previous task)
# ---------------------------------------------------------
baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train, y_train)
baseline_pred = baseline_model.predict(X_test)

baseline_acc = accuracy_score(y_test, baseline_pred)
baseline_precision = precision_score(y_test, baseline_pred)
baseline_recall = recall_score(y_test, baseline_pred)
baseline_f1 = f1_score(y_test, baseline_pred)

print("=" * 55)
print("BASELINE MODEL (default settings)")
print("=" * 55)
print(f"Accuracy:  {baseline_acc:.4f}")
print(f"Precision: {baseline_precision:.4f}")
print(f"Recall:    {baseline_recall:.4f}")
print(f"F1-score:  {baseline_f1:.4f}\n")
print("Classification Report:")
print(classification_report(y_test, baseline_pred, target_names=["Died", "Survived"]))

# ---------------------------------------------------------
# 3. Check class balance (why accuracy can mislead)
# ---------------------------------------------------------
print("Class balance in dataset:")
print(y.value_counts(normalize=True).rename({0: "Died", 1: "Survived"}))
print()

# ---------------------------------------------------------
# 4. Hyperparameter tuning with GridSearchCV
# ---------------------------------------------------------
# Tuning 2+ hyperparameters:
#   - C: inverse of regularization strength (smaller = stronger regularization)
#   - solver: optimization algorithm
#   - penalty: type of regularization
param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],
    "solver": ["liblinear", "lbfgs"]
}

grid_search = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000),
    param_grid=param_grid,
    cv=5,                # 5-fold cross-validation
    scoring="f1",         # optimize for F1 (better for imbalanced classes than accuracy)
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("=" * 55)
print("GRID SEARCH RESULTS")
print("=" * 55)
print(f"Best hyperparameters: {grid_search.best_params_}")
print(f"Best cross-validated F1-score: {grid_search.best_score_:.4f}\n")

# ---------------------------------------------------------
# 5. TUNED model evaluation
# ---------------------------------------------------------
best_model = grid_search.best_estimator_
tuned_pred = best_model.predict(X_test)

tuned_acc = accuracy_score(y_test, tuned_pred)
tuned_precision = precision_score(y_test, tuned_pred)
tuned_recall = recall_score(y_test, tuned_pred)
tuned_f1 = f1_score(y_test, tuned_pred)

print("=" * 55)
print("TUNED MODEL (best hyperparameters)")
print("=" * 55)
print(f"Accuracy:  {tuned_acc:.4f}")
print(f"Precision: {tuned_precision:.4f}")
print(f"Recall:    {tuned_recall:.4f}")
print(f"F1-score:  {tuned_f1:.4f}\n")
print("Classification Report:")
print(classification_report(y_test, tuned_pred, target_names=["Died", "Survived"]))

print("Confusion Matrix (Tuned Model):")
print(confusion_matrix(y_test, tuned_pred))
print()

# ---------------------------------------------------------
# 6. BEFORE / AFTER comparison table
# ---------------------------------------------------------
comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Baseline Model": [baseline_acc, baseline_precision, baseline_recall, baseline_f1],
    "Tuned Model": [tuned_acc, tuned_precision, tuned_recall, tuned_f1]
})
comparison["Improvement"] = comparison["Tuned Model"] - comparison["Baseline Model"]

print("=" * 55)
print("BEFORE vs AFTER COMPARISON")
print("=" * 55)
print(comparison.to_string(index=False))

BASELINE MODEL (default settings)
Accuracy:  0.8045
Precision: 0.7931
Recall:    0.6667
F1-score:  0.7244

Classification Report:
              precision    recall  f1-score   support

        Died       0.81      0.89      0.85       110
    Survived       0.79      0.67      0.72        69

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179

Class balance in dataset:
Survived
Died        0.616162
Survived    0.383838
Name: proportion, dtype: float64

GRID SEARCH RESULTS
Best hyperparameters: {'C': 0.1, 'solver': 'lbfgs'}
Best cross-validated F1-score: 0.7263

TUNED MODEL (best hyperparameters)
Accuracy:  0.7989
Precision: 0.8000
Recall:    0.6377
F1-score:  0.7097

Classification Report:
              precision    recall  f1-score   support

        Died       0.80      0.90      0.85       110
    Survived       0.80      0.64      0.71        69

    accuracy              

In [10]:
"""
Predict Titanic Survival — First Classification Model
--------------------------------------------------------
Loads the cleaned Titanic dataset, encodes categorical features,
splits into train/test sets, trains a Logistic Regression model,
and evaluates it with accuracy and a confusion matrix.
"""

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# ---------------------------------------------------------
# 1. Load data
# ---------------------------------------------------------
df = pd.read_csv("titanic.csv")

# ---------------------------------------------------------
# 2. Clean data (this should already be mostly done from your EDA step)
# ---------------------------------------------------------
# Fill missing Age with the median age
df["Age"] = df["Age"].fillna(df["Age"].median())

# Fill missing Embarked with the most common port
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# Drop columns that aren't useful for a first model
# (Name/Ticket/Cabin are high-cardinality/free text; PassengerId is just an index)
df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])

# ---------------------------------------------------------
# 3. Encode categorical columns
# ---------------------------------------------------------
df = pd.get_dummies(df, columns=["Sex", "Embarked"], drop_first=True)
# drop_first=True avoids the "dummy variable trap" (redundant columns)

# ---------------------------------------------------------
# 4. Split features/target and train/test sets
# ---------------------------------------------------------
X = df.drop(columns=["Survived"])
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---------------------------------------------------------
# 5. Train Logistic Regression model
# ---------------------------------------------------------
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# ---------------------------------------------------------
# 6. Evaluate
# ---------------------------------------------------------
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {acc:.4f}\n")

print("Confusion Matrix:")
print(cm)
print()
print("                Predicted: Died   Predicted: Survived")
print(f"Actual: Died        {cm[0][0]:>5}              {cm[0][1]:>5}")
print(f"Actual: Survived     {cm[1][0]:>5}              {cm[1][1]:>5}")
print()

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Died", "Survived"]))

# ---------------------------------------------------------
# 7. Feature coefficients (bonus: which features matter most)
# ---------------------------------------------------------
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
}).sort_values("Coefficient", key=abs, ascending=False)

print("Feature importance (Logistic Regression coefficients):")
print(coef_df.to_string(index=False))


Accuracy: 0.8045

Confusion Matrix:
[[98 12]
 [23 46]]

                Predicted: Died   Predicted: Survived
Actual: Died           98                 12
Actual: Survived        23                 46

Classification Report:
              precision    recall  f1-score   support

        Died       0.81      0.89      0.85       110
    Survived       0.79      0.67      0.72        69

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179

Feature importance (Logistic Regression coefficients):
   Feature  Coefficient
  Sex_male    -2.555755
    Pclass    -1.090476
Embarked_S    -0.381616
Embarked_Q     0.278320
     SibSp    -0.243809
     Parch    -0.071496
       Age    -0.038464
      Fare     0.002255
